1. 도구(Tool) 준비: 외부와 소통하는 일반 파이썬 함수

가장 먼저 LLM이 사용할 기능(함수)을 정의합니다. 여기서 중요한 것은 @tool 데코레이터와 함수 내부의 """설명(Docstring)"""입니다.

In [ ]:
import os
import requests
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.errors import GraphRecursionError

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

# [함수 1] 영화 검색기
@tool
def search_movies(query: str) -> str:
    """영화를 검색한다. 영화 제목을 검색할 수 있다. 예: '인셉션', '기생충', '아이언맨'"""
    # 순수 파이썬 로직: TMDB API를 호출해서 검색 결과를 문자열로 반환
    response = requests.get(
        "https://api.themoviedb.org/3/search/movie",
        headers={"Authorization": f"Bearer {TMDB_API_KEY}"},
        params={"query": query, "language": "ko-KR"},
    )
    data = response.json()
    results = data.get("results", [])[:5]
    
    if not results:
        return f"'{query}'에 대한 검색 결과가 없습니다."

    output = []
    for movie in results:
        movie_id = movie.get("id")
        title = movie.get("title", "제목 없음")
        output.append(f"- [{movie_id}] {title}")
    return "\n".join(output)

# [함수 2] 영화 상세 정보 조회기
@tool
def get_movie_detail(movie_id: int) -> str:
    """영화의 상세정보를 조회한다. movie_id에는 영화의 id가 들어간다."""
    # 순수 파이썬 로직: ID를 가지고 영화의 세부 정보를 가져옴
    response = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}",
        headers={"Authorization": f"Bearer {TMDB_API_KEY}"},
        params={"language": "ko-KR"},
    )
    return str(response.json())

# LLM에게 쥐어줄 도구 리스트
tools = [search_movies, get_movie_detail]